In [ ]:
!pip install transformers datasets evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.0 MB/s eta 0:00:00


In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split
import evaluate



In [ ]:
path_to_train_file = tf.keras.utils.get_file(
    'ratings_train.txt',
    'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt'
)

# TSV 파일 읽기
train_df = pd.read_csv(path_to_train_file, sep='\t')

# 필요한 컬럼만 사용
train_df = train_df[['document', 'label']]

# 결측값 제거
train_df = train_df.dropna()

# 학습 속도를 위해 일부 데이터만 사용
train_df = train_df[:10000]

print(train_df.head())


14628807/14628807 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
                                            document  label
0                                아 더빙.. 진짜 짜증나네요 목소리      0
1                  흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나      1
2                                  너무재밓었다그래서보는것을추천한다      0
3                      교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정      0
4  사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...      1


In [ ]:
texts = train_df['document'].tolist()
labels = train_df['label'].tolist()

print(texts[:3])
print(labels[:3])


['아 더빙.. 진짜 짜증나네요 목소리', '흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나', '너무재밓었다그래서보는것을추천한다']
[0, 1, 0]


In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42
)


In [ ]:
train_dataset = Dataset.from_dict({
    'text': train_texts,
    'label': train_labels
})

val_dataset = Dataset.from_dict({
    'text': val_texts,
    'label': val_labels
})

print(train_dataset)
print(val_dataset)


Dataset({
    features: ['text', 'label'],
    num_rows: 8000
})
Dataset({
    features: ['text', 'label'],
    num_rows: 2000
})


In [ ]:
MODEL_NAME = "klue/bert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on you

In [ ]:
MAX_LENGTH = 64

def tokenize_function(example):

    return tokenizer(
        example['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH
    )


# tokenizing 적용
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)



Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
train_dataset.set_format(
    type='torch',
    columns=[
        'input_ids',
        'attention_mask',
        'label'
    ]
)

val_dataset.set_format(
    type='torch',
    columns=[
        'input_ids',
        'attention_mask',
        'label'
    ]
)

In [ ]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=2,

    weight_decay=0.01,

    logging_dir="./logs",

    logging_steps=50
)



`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    compute_metrics=compute_metrics
)


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.351440,0.389660,0.849000
2,0.259918,0.541063,0.860500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2000, training_loss=0.310781320810318, metrics={'train_runtime': 267.4905, 'train_samples_per_second': 59.815, 'train_steps_per_second': 7.477, 'total_flos': 526222110720000.0, 'train_loss': 0.310781320810318, 'epoch': 2.0})

In [ ]:
results = trainer.evaluate()

print("\nValidation Results")
print(results)




Validation Results
{'eval_loss': 0.5410630106925964, 'eval_accuracy': 0.8605, 'eval_runtime': 7.5948, 'eval_samples_per_second': 263.339, 'eval_steps_per_second': 32.917, 'epoch': 2.0}


In [ ]:
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 모델 GPU로 이동
model.to(device)

test_sentences = [
    "이 영화는 참신하긴 한데 배우들의 연기력이 좀 아쉬움",
    "완전 지루하고 졸렸음 돈 아까움 나도 영화 찍겠네",
    "좋아하는 배우가 나와 재미있게 봤음!"
]

# 토큰화
inputs = tokenizer(
    test_sentences,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=64
)

# 입력 데이터도 GPU로 이동
inputs = {k: v.to(device) for k, v in inputs.items()}

# 예측
with torch.no_grad():
    outputs = model(**inputs)

predictions = np.argmax(
    outputs.logits.cpu().numpy(),
    axis=1
)

# 결과 출력
for sentence, pred in zip(test_sentences, predictions):
    label = "긍정 😊" if pred == 1 else "부정 😞"

    print(f"문장: {sentence}")
    print(f"예측 결과: {label}")
    print("-" * 30)

문장: 이 영화는 참신하긴 한데 배우들의 연기력이 좀 아쉬움
예측 결과: 부정 😞
------------------------------
문장: 완전 지루하고 졸렸음 돈 아까움 나도 영화 찍겠네
예측 결과: 부정 😞
------------------------------
문장: 좋아하는 배우가 나와 재미있게 봤음!
예측 결과: 긍정 😊
------------------------------


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
MY_MODEL_NAME = "sehee77/nsmc-sentiment"

model.push_to_hub(MY_MODEL_NAME)
tokenizer.push_to_hub(MY_MODEL_NAME)

print("업로드 완료!")
print(f"모델 주소: https://huggingface.co/{MY_MODEL_NAME}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...3o0v1bm/model.safetensors:   0%|          |  556kB /  442MB            

README.md: 0.00B [00:00, ?B/s]

업로드 완료!
모델 주소: https://huggingface.co/sehee77/nsmc-sentiment


In [ ]:
from huggingface_hub import HfApi, create_repo

MY_MODEL_NAME = "sehee77/nsmc-sentiment"
MY_SPACE_NAME = "sehee77/spacename"

create_repo(
    repo_id=MY_SPACE_NAME,
    repo_type="space",
    space_sdk="gradio",
    exist_ok=True
)

app_code = f'''
import gradio as gr
from transformers import pipeline

classifier = pipeline("text-classification", model="{MY_MODEL_NAME}")

def format_result(result):
    label = result["label"]
    score = result["score"]

    if label == "LABEL_1":
        emoji, label_kr = "😊", "긍정"
    else:
        emoji, label_kr = "😞", "부정"

    return f"{{emoji}} {{label_kr}} (확신도: {{score:.1%}})"

def predict(text):
    if not text.strip():
        return "문장을 입력해주세요."
    result = classifier(text)[0]
    return format_result(result)

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(label="영화 리뷰", placeholder="리뷰를 입력하세요...", lines=3),
    outputs=gr.Textbox(label="감정 분석 결과"),
    title="AI 영화 리뷰 감정 분석기",
    description="NSMC 데이터로 파인튜닝된 한국어 감정 분석 모델입니다.",
    examples=[
        ["이 영화 진짜 재미있어요!"],
        ["완전 지루하고 별로였음"],
        ["배우 연기는 좋았지만 스토리가 아쉬웠다"]
    ]
)
demo.launch()
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

with open('requirements.txt', 'w') as f:
    f.write("transformers\ngradio\ntorch\n")

api = HfApi()
api.upload_file(
    path_or_fileobj="app.py",
    path_in_repo="app.py",
    repo_id=MY_SPACE_NAME,
    repo_type="space"
)
api.upload_file(
    path_or_fileobj="requirements.txt",
    path_in_repo="requirements.txt",
    repo_id=MY_SPACE_NAME,
    repo_type="space"
)

print("완료!")
print(f"https://huggingface.co/spaces/{MY_SPACE_NAME}")

완료!
https://huggingface.co/spaces/sehee77/spacename
